In [1]:
# Imports and Data Loading

import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

# Load the dataset
df = pd.read_csv('resume_dataset.csv')
print(df.head())

         Category                                             Resume
0  Data Scientist  Senior Data Scientist with 7+ years of experie...
1  Data Scientist  Junior Data Scientist with a passion for NLP a...
2  Data Scientist  Data Scientist with 5 years in fintech. Expert...
3  Data Scientist  Analytical Data Scientist with a focus on mark...
4  Data Scientist  Research-oriented Data Scientist with a PhD in...


In [2]:
# Data Preprocessing

def clean_resume(resume_text):
    # Remove URLs, hashtags, mentions, special characters
    resume_text = re.sub('http\S+\s*', ' ', resume_text)
    resume_text = re.sub('RT|cc', ' ', resume_text)
    resume_text = re.sub('#\S+', '', resume_text)
    resume_text = re.sub('@\S+', '  ', resume_text)
    resume_text = re.sub('[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', resume_text)
    resume_text = re.sub(r'[^\x00-\x7f]',r' ', resume_text)
    resume_text = re.sub('\s+', ' ', resume_text)
    return resume_text.lower()

df['Cleaned_Resume'] = df['Resume'].apply(lambda x: clean_resume(x))

In [3]:
# Vectorization and Model Training

# Convert text to numerical vectors
tfidf = TfidfVectorizer(stop_words='english')
X = tfidf.fit_transform(df['Cleaned_Resume'])
y = df['Category']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train KNN Model
clf = KNeighborsClassifier(n_neighbors=3)
clf.fit(X_train, y_train)

# Evaluate
print(classification_report(y_test, clf.predict(X_test)))

                precision    recall  f1-score   support

      Advocate       0.83      1.00      0.91         5
Data Scientist       1.00      1.00      1.00         6
 HR Specialist       1.00      0.67      0.80         3
 Web Developer       1.00      1.00      1.00         6

      accuracy                           0.95        20
     macro avg       0.96      0.92      0.93        20
  weighted avg       0.96      0.95      0.95        20



In [4]:
# Prediction Function

def predict_category(resume_text):
    cleaned = clean_resume(resume_text)
    vectorized = tfidf.transform([cleaned])
    return clf.predict(vectorized)[0]

# Test it
sample_resume = "Expert in Python, machine learning, and deep learning models."
print(f"Predicted Category: {predict_category(sample_resume)}")

Predicted Category: Data Scientist
